# Pre-M0.8 — IQ Axes, Power, and Capstone Check

**Unit objective.** Verify that you understand the IQ array convention (axis 0 = examples, axis 1 = I/Q, axis 2 = time samples), can compute and compare power using two independent methods, and can detect, explain, and correct an injected axis-swap error — the single most common mistake when working with IQ data.

**What you should learn from this notebook.**

1. Identify and explain what each axis of an `(N, 2, L)` IQ array represents.
2. Compute signal power via the real-valued formula `P = mean(I² + Q²)` and via the complex-valued formula `P = mean(|z|²)`.
3. Detect an axis-swap error by inspecting array shapes.
4. Correct an axis-swap using `np.transpose` or equivalent.
5. Confirm that both power methods agree within numerical tolerance.
6. Combine every concept from notebooks 1–7 into a single capstone exercise.

**IQ data used.** Synthetic IQ vectors generated with `SEED = 42` inside this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)

N = 200       # number of examples
L = 512       # time samples per example

# Canonical layout: X.shape == (N, 2, L)
X = np.zeros((N, 2, L), dtype=np.float32)
X[:, 0, :] = rng.standard_normal((N, L)).astype(np.float32)  # I
X[:, 1, :] = rng.standard_normal((N, L)).astype(np.float32)  # Q

print(f"X.shape  = {X.shape}")
print(f"X.dtype  = {X.dtype}")
print(f"X[:,0,:] range: [{X[:,0,:].min():.4f}, {X[:,0,:].max():.4f}]")
print(f"X[:,1,:] range: [{X[:,1,:].min():.4f}, {X[:,1,:].max():.4f}]")

---

## 1  Review of Key Concepts

### 1.1  The three axes

| Axis | Dimension | Meaning |
|------|-----------|----------|
| 0 | `N` | example index |
| 1 | 2 | component: 0 = I, 1 = Q |
| 2 | `L` | time-sample index |

In [ ]:
# Axis 0: select example 0
print(f"X[0].shape       = {X[0].shape}")       # (2, L)

# Axis 1: select I component for all examples
print(f"X[:, 0, :].shape = {X[:, 0, :].shape}") # (N, L)

# Axis 2: select time-sample 0 for all examples
print(f"X[:, :, 0].shape = {X[:, :, 0].shape}") # (N, 2)

### 1.2  Indexing and broadcasting

NumPy broadcasts along dimensions of length 1 (or missing dimensions). Broadcasting a `(L,)` array against `(N, L)` works because the trailing dimension matches.

In [ ]:
# Broadcasting example: subtract mean per component
# X[:, :, :] has shape (N, 2, L)
# mean over axis=2 keeps dims: (N, 2, 1)
X_mean = np.mean(X, axis=2, keepdims=True)       # (N, 2, 1)
X_centered = X - X_mean                           # broadcasts (N, 2, L) - (N, 2, 1)
print(f"X_centered.shape = {X_centered.shape}")    # (N, 2, L)
print(f"Mean of X_centered per example ~ 0: {np.allclose(X_centered.mean(axis=2), 0)}")

### 1.3  IQ power — two equivalent methods

**Method A (real-valued):** $P = \text{mean}(I^2 + Q^2)$

**Method B (complex-valued):** $z = I + jQ$, $P = \text{mean}(|z|^2)$

In [ ]:
I = X[:, 0, :]  # (N, L)
Q = X[:, 1, :]  # (N, L)

# Method A
P_iq = np.mean(I**2 + Q**2)

# Method B
z = I + 1j * Q
P_complex = np.mean(np.abs(z)**2)

print(f"Method A (real)  : P_iq     = {P_iq:.10f}")
print(f"Method B (complex): P_complex = {P_complex:.10f}")
print(f"Difference        : {abs(P_iq - P_complex):.2e}")

---

## 2  Comprehensive Exercises

### Exercise 1 — Axis identification

Given `X.shape == (10, 2, 1024)`, answer:
1. What does `X[3, :, :]` return and what is its shape?
2. What does `X[:, 1, :]` return and what does axis 1 == 1 mean?
3. What does `X[:, :, 99]` return?

In [ ]:
X_ex1 = np.zeros((10, 2, 1024), dtype=np.float32)

# STUDENT ATTEMPT
# Write your answers as shape + brief description comments,
# then fill in the variables below.

# shape_q1 = (??? , ??? , ??? )  # fill in
# shape_q2 = (??? , ??? , ??? )
# shape_q3 = (??? , ??? , ??? )

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
# shape_q1 = (2, 1024)      # example 3, both I and Q, all time samples
# shape_q2 = (10, 1024)     # all examples, Q component, all time samples
# shape_q3 = (10, 2)        # all examples, both components, time-sample 99

### Exercise 2 — Select I, compute power per example

Extract I and Q from `X` (the canonical array from the setup cell), then compute the power of each example independently using Method A. Store the result in `powers_per_ex` with shape `(N,)`.

In [ ]:
# STUDENT ATTEMPT
# I_per_ex = X[:, 0, :]
# Q_per_ex = X[:, 1, :]
# powers_per_ex = ??

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
I_per_ex = X[:, 0, :]
Q_per_ex = X[:, 1, :]
powers_per_ex = np.mean(I_per_ex**2 + Q_per_ex**2, axis=1)
print(f"powers_per_ex.shape = {powers_per_ex.shape}")   # (200,)
print(f"powers_per_ex[:5]   = {powers_per_ex[:5]}")

### Exercise 3 — Broadcasting: normalize each example

Normalize each example in `X` by dividing by its own power so that each example has unit power. Use broadcasting — no loops. Store in `X_norm` with the same shape and dtype as `X`.

In [ ]:
# STUDENT ATTEMPT
# X_norm = ??

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
X_norm = X / np.sqrt(powers_per_ex[:, None, None]).astype(np.float32)
print(f"X_norm.shape = {X_norm.shape}")
check_power = np.mean(X_norm[:, 0, :]**2 + X_norm[:, 1, :]**2, axis=1)
print(f"Power per example after normalization (first 5): {check_power[:5]}")
print(f"All ~ 1.0? {np.allclose(check_power, 1.0, atol=1e-5)}")

### Exercise 4 — Per-example power agreement

Compute per-example power with both Method A and Method B. Verify they agree for every example within tolerance `1e-5`.

In [ ]:
# STUDENT ATTEMPT
# P_per_ex_A = ??
# P_per_ex_B = ??
# agreement_per_ex = ??

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
P_per_ex_A = np.mean(X[:, 0, :]**2 + X[:, 1, :]**2, axis=1)
z_per_ex = X[:, 0, :] + 1j * X[:, 1, :]
P_per_ex_B = np.mean(np.abs(z_per_ex)**2, axis=1)
agreement_per_ex = np.allclose(P_per_ex_A, P_per_ex_B, rtol=1e-5)
print(f"Per-example power agreement: {agreement_per_ex}")

### Exercise 5 — Plot I and Q for one example

Plot the I and Q components of example 0 as two separate lines in the same figure.

In [ ]:
# STUDENT ATTEMPT
# plt.figure(figsize=(10, 3))
# ...
# plt.show()

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
plt.figure(figsize=(10, 3))
plt.plot(X[0, 0, :], label="I", alpha=0.8)
plt.plot(X[0, 1, :], label="Q", alpha=0.8)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Example 0 — I and Q")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 6 — Scatter plot I vs Q

Create an I-vs-Q scatter plot for example 0. Use a small marker size and alpha for visibility.

In [ ]:
# STUDENT ATTEMPT
# plt.figure(figsize=(5, 5))
# ...
# plt.show()

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
plt.figure(figsize=(5, 5))
plt.scatter(X[0, 0, :], X[0, 1, :], s=1, alpha=0.3)
plt.xlabel("I")
plt.ylabel("Q")
plt.title("I vs Q — Example 0")
plt.axis("equal")
plt.tight_layout()
plt.show()

---

## 3  Capstone Challenge

Apply everything you have learned. Follow the instructions in order.

**3.1** Create `Y`, a new IQ array of shape `(N, 2, L)` where:
- `Y[:, 0, :]` is a cosine wave with a different frequency per example,
- `Y[:, 1, :]` is the corresponding sine wave.

Use `np.arange` for the time axis and `np.cos` / `np.sin` for the signals. The frequencies should be drawn from a uniform distribution between 0.01 and 0.1 (cycles per sample) using `rng`.

In [ ]:
# STUDENT ATTEMPT
# t = np.arange(L, dtype=np.float32)                  # (L,)
# freqs = rng.uniform(0.01, 0.1, size=N).astype(np.float32)  # (N,)
# Y = np.zeros((N, 2, L), dtype=np.float32)
# Y[:, 0, :] = ...
# Y[:, 1, :] = ...

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
t = np.arange(L, dtype=np.float32)
freqs = rng.uniform(0.01, 0.1, size=N).astype(np.float32)
Y = np.zeros((N, 2, L), dtype=np.float32)
Y[:, 0, :] = np.cos(2 * np.pi * freqs[:, None] * t[None, :]).astype(np.float32)
Y[:, 1, :] = np.sin(2 * np.pi * freqs[:, None] * t[None, :]).astype(np.float32)
print(f"Y.shape = {Y.shape}, Y.dtype = {Y.dtype}")

**3.2** Compute the total power of each example in `Y` using Method A. Then verify that for a pure cosine/sine pair the power per example should be approximately 0.5.

In [ ]:
# STUDENT ATTEMPT
# P_Y = ??
# print(f"Power of first 5 examples: {P_Y[:5]}")
# print(f"Expected ~ 0.5")

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
P_Y = np.mean(Y[:, 0, :]**2 + Y[:, 1, :]**2, axis=1)
print(f"Power of first 5 examples: {P_Y[:5]}")
print(f"Mean power across all examples: {P_Y.mean():.6f}")
print(f"All ~ 0.5? {np.allclose(P_Y, 0.5, atol=1e-5)}")

**3.3** Combine `X` and `Y` along axis 0 into a single array `XY` of shape `(2N, 2, L)`. Then compute the mean power across all examples. Verify the result is the average of the mean power of `X` and the mean power of `Y`.

In [ ]:
# STUDENT ATTEMPT
# XY = ??
# P_XY = ??
# P_X_mean = ??
# P_Y_mean = ??
# expected_mean = ??

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
XY = np.concatenate([X, Y], axis=0)
P_XY = np.mean(XY[:, 0, :]**2 + XY[:, 1, :]**2)
P_X_mean = np.mean(X[:, 0, :]**2 + X[:, 1, :]**2)
P_Y_mean = np.mean(Y[:, 0, :]**2 + Y[:, 1, :]**2)
expected_mean = (P_X_mean + P_Y_mean) / 2
print(f"XY.shape = {XY.shape}")
print(f"P_XY   = {P_XY:.10f}")
print(f"Expected = {expected_mean:.10f}")
print(f"Agree? {np.allclose(P_XY, expected_mean, atol=1e-5)}")

---

## 4  Pass Criterion Challenge (PC-1, PC-2, PC-3)

### STUDENT AXIS EXPLANATION — PC-1

**STUDENT ATTEMPT**

Answer the following six questions in the Markdown cell below. Write your answers before looking at any solution.

1. Given `X.shape == (N, 2, L)`, what does axis 0 represent?
2. What does axis 1 represent, and what are its two possible values?
3. What does axis 2 represent?
4. If you run `X[:, 0, :]`, what is the resulting shape and what component does it contain?
5. If you run `X[:, :, 0]`, what is the resulting shape and what does each row contain?
6. Explain why the ordering `(N, 2, L)` is preferred over `(N, L, 2)` for IQ data.

**Write your answers here:**

1. 

2. 

3. 

4. 

5. 

6. 

In [ ]:
# Student/instructor must manually change this to True after verifying answers.
AXES_EXPLANATION_VERIFIED = False
assert isinstance(AXES_EXPLANATION_VERIFIED, bool)

**OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT**

1. Axis 0 is the **example axis**. Each index along axis 0 selects one IQ example (one waveform).
2. Axis 1 is the **component axis**. It has exactly two values: 0 = I (in-phase), 1 = Q (quadrature).
3. Axis 2 is the **time-sample axis**. Each index selects one time sample within the waveform.
4. `X[:, 0, :]` has shape `(N, L)` and contains the **I component** of every example.
5. `X[:, :, 0]` has shape `(N, 2)` and each row contains the **(I, Q) pair at time-sample 0** for that example.
6. With `(N, 2, L)`, slicing `X[:, 0, :]` gives a contiguous `(N, L)` block for I — a single component across all examples and time samples. This is the natural layout for operations like power computation (`mean(I^2 + Q^2)` over axis 2) and avoids reordering when extracting components.

---

### INJECTED AXIS SWAP — PC-2

In [ ]:
# --- Deliberately injected axis-swap error ---
# This simulates the most common IQ data bug:
# someone saved data as (N, L, 2) instead of (N, 2, L).
X_swapped = np.transpose(X, (0, 2, 1))
print(f"X_swapped.shape = {X_swapped.shape}")  # (N, L, 2) — WRONG!

**STUDENT ATTEMPT**

Inspect `X_swapped` and answer:
1. What is the shape of `X_swapped`?
2. Which axis currently holds I/Q, and is that correct?
3. Why is this a problem for IQ processing?

Then write code to fix it: produce `X_fixed` with shape `(N, 2, L)` and dtype `float32`, equal to the original `X`.

In [ ]:
# STUDENT ATTEMPT
# X_fixed = ??

# OPTIONAL SOLUTION — REVEAL ONLY AFTER ATTEMPT
# The original X had I/Q on axis 1 and time on axis 2.
# X_swapped swapped those two axes, so axis 1 now holds time and axis 2 holds I/Q.
# To fix: swap axes 1 and 2 back.
X_fixed = np.transpose(X_swapped, (0, 2, 1)).astype(np.float32)
print(f"X_fixed.shape = {X_fixed.shape}")

In [ ]:
# --- Validation (PC-2) ---
axis_swap_corrected = (
    X_fixed.shape == X.shape
    and X_fixed.dtype == X.dtype
    and np.array_equal(X_fixed, X)
)
assert X_fixed.shape == X.shape, f"Shape mismatch: {X_fixed.shape} != {X.shape}"
assert X_fixed.dtype == X.dtype, f"Dtype mismatch: {X_fixed.dtype} != {X.dtype}"
assert np.array_equal(X_fixed, X), "Values do not match original X"
print("PC-2 PASS: axis swap correctly recovered.")

---

### IQ POWER AGREEMENT — PC-3

In [ ]:
# --- Independent power computation (two methods) ---

# Method A: real-valued
I_pc3 = X[:, 0, :]  # (N, L)
Q_pc3 = X[:, 1, :]  # (N, L)
P_iq = np.mean(I_pc3**2 + Q_pc3**2)

# Method B: complex-valued
z = I_pc3 + 1j * Q_pc3
P_complex = np.mean(np.abs(z)**2)

print(f"Method A — P_iq     = {P_iq:.10f}")
print(f"Method B — P_complex = {P_complex:.10f}")

In [ ]:
# --- Validation (PC-3) ---
POWER_RTOL = 1e-5
POWER_ATOL = 1e-7

power_consistency = np.allclose(P_iq, P_complex, rtol=POWER_RTOL, atol=POWER_ATOL)
assert power_consistency, (
    f"Power mismatch: P_iq={P_iq}, P_complex={P_complex}, "
    f"diff={abs(P_iq - P_complex):.2e}"
)
print("PC-3 PASS: both power methods agree within tolerance.")

---

## 5  Automatic Validations

In [ ]:
# Gate status tracking
pc1_pass = bool(AXES_EXPLANATION_VERIFIED)
pc2_pass = bool(axis_swap_corrected)
pc3_pass = bool(power_consistency)

print(f"PC-1 (Axis Explanation) : {'PASS' if pc1_pass else 'WAIT'}")
print(f"PC-2 (Axis Swap Fix)    : {'PASS' if pc2_pass else 'WAIT'}")
print(f"PC-3 (Power Agreement)  : {'PASS' if pc3_pass else 'WAIT'}")

## 6  Manual Evaluation of Axis Explanation

The instructor (or student, after attempting) should verify the six written answers in the PC-1 section above. The `AXES_EXPLANATION_VERIFIED` flag must be changed to `True` manually — it is never set automatically.

---

## 7  PASS CRITERION GATE

In [ ]:
pc1_status = "PASS" if pc1_pass else "WAIT"
pc2_status = "PASS" if pc2_pass else "WAIT"
pc3_status = "PASS" if pc3_pass else "WAIT"

print(f"PC-1 Axis Explanation : {pc1_status}")
print(f"PC-2 Axis Swap Fix    : {pc2_status}")
print(f"PC-3 Power Agreement  : {pc3_status}")
print()

all_pass = pc1_pass and pc2_pass and pc3_pass
if all_pass:
    print("PRE-M0.8 FINAL STATUS: PASS")
else:
    print("PRE-M0.8 FINAL STATUS: WAIT")